<a href="https://colab.research.google.com/github/import-dhruv/AI-Research/blob/main/multi_turn_dataset_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Multi-Turn Dataset Pipeline — Fixed & Completed
Fixes:
  1. MultiClassSFTDataset method indentation (were nested inside __init__)
  2. item['message'] → item['messages']
  3. content = msg['role'] → content = msg['content']
  4. self.sampples → self.samples
  5. truncate_context now copies item to avoid mutation
  6. Added collate_fn for DataLoader batching
  7. Added sample data.jsonl creation so the pipeline can run end-to-end
"""

# ── 0. Install deps (Colab / script) ──────────────────────────────────────────
# Uncomment the line below when running in Colab:
# !pip install datasets pandas tqdm presidio-analyzer presidio-anonymizer transformers torch

import json
import copy
import hashlib
from typing import List, Dict, Any
from tqdm import tqdm
from datasets import Dataset, DatasetDict


# ── 1. Core Pipeline ──────────────────────────────────────────────────────────

class MultiTurnPipeline:
    def __init__(self, min_turns: int = 3, max_turns: int = 20, max_tokens: int = 4096):
        self.min_turns = min_turns
        self.max_turns = max_turns
        self.max_tokens = max_tokens

    def load_raw_data(self, file_path: str) -> List[Dict]:
        """Load JSON or JSONL file."""
        data = []
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()

        # Try as a JSON array / object first
        try:
            parsed = json.loads(content)
            if isinstance(parsed, dict):
                return [parsed]
            elif isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass

        # Fall back to JSONL (one JSON object per line)
        for line in content.splitlines():
            line = line.strip()
            if line:
                data.append(json.loads(line))
        return data

    def normalize_schema(self, raw_data: List[Dict]) -> List[Dict]:
        """Normalise varying field names into a standard messages schema."""
        normalized = []
        for item in raw_data:
            messages = item.get('conversations', item.get('messages', []))

            standard_msgs = []
            for msg in messages:
                role = msg.get('role', msg.get('from', 'user'))
                if role == 'human':
                    role = 'user'
                if role == 'gpt':
                    role = 'assistant'

                content = msg.get('content', msg.get('value', ''))
                if isinstance(content, list):
                    content = ' '.join(str(c) for c in content)

                standard_msgs.append({"role": role, "content": content})

            # Ensure a system message is always first
            if standard_msgs and standard_msgs[0]['role'] != 'system':
                standard_msgs.insert(0, {"role": "system", "content": "You are a helpful assistant."})

            normalized.append({
                "id": item.get('id', self._generate_id(standard_msgs)),
                "messages": standard_msgs,
                "metadata": item.get('metadata', {})
            })
        return normalized

    def filter_quality(self, data: List[Dict]) -> List[Dict]:
        """Drop low-quality conversations."""
        filtered = []
        for item in data:
            msgs = item['messages']

            # Check turn count (system message excluded)
            dialogue_turns = [m for m in msgs if m['role'] in ['user', 'assistant']]
            if len(dialogue_turns) < self.min_turns * 2:
                continue

            # Drop conversations with very short messages
            if any(len(m['content'].strip()) < 5 for m in msgs):
                continue

            # Drop highly repetitive conversations
            contents = [m['content'] for m in msgs]
            if len(set(contents)) < len(contents) * 0.5:
                continue

            filtered.append(item)
        return filtered

    def truncate_context(self, data: List[Dict]) -> List[Dict]:
        """Trim overly long conversations from the front, preserving system msg."""
        processed = []
        for item in data:
            item = copy.deepcopy(item)          # FIX: avoid mutating original
            msgs = item['messages']

            system_msg = [msgs[0]] if msgs and msgs[0]['role'] == 'system' else []
            dialogue = msgs[len(system_msg):]

            if len(dialogue) > self.max_turns * 2:
                dialogue = dialogue[-(self.max_turns * 2):]

            item['messages'] = system_msg + dialogue
            processed.append(item)
        return processed

    def _generate_id(self, messages: List[Dict]) -> str:
        content = "".join(m['content'] for m in messages)
        return hashlib.md5(content.encode()).hexdigest()

    def run(self, input_path: str, output_path: str):
        print("Loading data...")
        raw = self.load_raw_data(input_path)

        print("Normalizing schema...")
        normalized = self.normalize_schema(raw)

        print("Filtering quality...")
        filtered = self.filter_quality(normalized)
        print(f"   Retained {len(filtered)}/{len(normalized)} samples")

        print("Truncating context...")
        final = self.truncate_context(filtered)

        # Save to JSONL
        with open(output_path, 'w', encoding='utf-8') as f:
            for item in final:
                f.write(json.dumps(item) + '\n')
        print(f"Pipeline complete. Saved {len(final)} samples to {output_path}")
        return final


# ── 2. PyTorch Dataset ────────────────────────────────────────────────────────

import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import AutoTokenizer


class MultiTurnSFTDataset(TorchDataset):
    """
    Tokenises processed JSONL for supervised fine-tuning.
    Labels are -100 everywhere except assistant turns (standard SFT masking).
    """

    def __init__(self, data_path: str, tokenizer: AutoTokenizer, max_length: int = 2048):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = self._load_and_tokenize(data_path)   # FIX: call as normal method

    # FIX: all methods at correct indentation level (class, not __init__)
    def _load_and_tokenize(self, path: str):
        samples = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                item = json.loads(line)
                messages = item['messages']                  # FIX: was item['message']

                # Apply chat template if available, else fall back to manual join
                try:
                    text = self.tokenizer.apply_chat_template(
                        messages, tokenize=False, add_generation_prompt=False
                    )
                except Exception:
                    text = "\n".join(f"{m['role']}: {m['content']}" for m in messages)

                tokens = self.tokenizer(
                    text,
                    add_special_tokens=True,
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors=None,
                )['input_ids']

                # Build labels: mask everything, then unmask assistant turns
                labels = [-100] * len(tokens)
                current_idx = 0

                for msg in messages:
                    role = msg['role']
                    content = msg['content']                 # FIX: was msg['role']
                    content_tokens = self.tokenizer(
                        content,
                        add_special_tokens=False,
                        return_tensors=None,
                    )['input_ids']
                    content_len = len(content_tokens)

                    if role == 'assistant':
                        end = min(current_idx + content_len, len(tokens))
                        labels[current_idx:end] = tokens[current_idx:end]

                    current_idx = min(current_idx + content_len + 5, len(tokens))

                samples.append({
                    "input_ids": tokens,
                    "attention_mask": [1] * len(tokens),
                    "labels": labels,
                })

        return samples

    def __len__(self):
        return len(self.samples)                             # FIX: was self.sampples

    def __getitem__(self, idx):
        return {k: torch.tensor(v) for k, v in self.samples[idx].items()}


def collate_fn(batch, pad_token_id: int = 0):
    """Pad sequences in a batch to the same length."""
    max_len = max(item['input_ids'].size(0) for item in batch)

    padded = {"input_ids": [], "attention_mask": [], "labels": []}
    for item in batch:
        seq_len = item['input_ids'].size(0)
        pad_len = max_len - seq_len

        padded['input_ids'].append(
            torch.cat([item['input_ids'], torch.full((pad_len,), pad_token_id)])
        )
        padded['attention_mask'].append(
            torch.cat([item['attention_mask'], torch.zeros(pad_len, dtype=torch.long)])
        )
        padded['labels'].append(
            torch.cat([item['labels'], torch.full((pad_len,), -100)])
        )

    return {k: torch.stack(v) for k, v in padded.items()}


# ── 3. Validation helper ──────────────────────────────────────────────────────

def validate_conversation(messages: List[Dict]) -> tuple:
    roles = [m['role'] for m in messages if m['role'] != 'system']
    for i in range(len(roles) - 1):
        if roles[i] == roles[i + 1]:
            return False, f"Consecutive '{roles[i]}' roles at position {i}"
    return True, "OK"


# ── 4. Expansion prompt template ──────────────────────────────────────────────

EXPANSION_PROMPT = """
Convert this single Q&A pair into a 3-turn conversation.
- Turn 1: User asks the original question.
- Turn 2: Assistant asks a clarifying question.
- Turn 3: User answers clarification.
- Turn 4: Assistant gives final answer.

Original Q: {question}
Original A: {answer}
"""


# ── 5. End-to-end demo ────────────────────────────────────────────────────────

def create_sample_data(path: str):
    """Write a minimal data.jsonl so the pipeline can run without real data."""
    samples = [
        {
            "id": "sample_001",
            "conversations": [
                {"role": "user",      "content": "Can you help me understand recursion in Python?"},
                {"role": "assistant", "content": "Sure! Recursion is when a function calls itself. Would you like a simple example?"},
                {"role": "user",      "content": "Yes please, something with factorials."},
                {"role": "assistant", "content": "Great choice! Here's a factorial function:\n\ndef factorial(n):\n    if n == 0:\n        return 1\n    return n * factorial(n - 1)\n\nfactorial(5) returns 120."},
                {"role": "user",      "content": "What happens if I pass a negative number?"},
                {"role": "assistant", "content": "Without a guard clause it will recurse infinitely and raise RecursionError. Add `if n < 0: raise ValueError('n must be >= 0')` at the top."},
            ]
        },
        {
            "id": "sample_002",
            "conversations": [
                {"role": "user",      "content": "What is gradient descent?"},
                {"role": "assistant", "content": "Gradient descent is an optimisation algorithm that iteratively adjusts parameters to minimise a loss function by moving in the direction of the steepest descent."},
                {"role": "user",      "content": "How does learning rate affect it?"},
                {"role": "assistant", "content": "A high learning rate may overshoot the minimum; a low rate converges slowly. Adaptive methods like Adam adjust the rate automatically."},
                {"role": "user",      "content": "What is Adam exactly?"},
                {"role": "assistant", "content": "Adam (Adaptive Moment Estimation) combines momentum and RMSProp. It maintains per-parameter learning rates adapted from first and second moment estimates of the gradients."},
            ]
        },
    ]
    with open(path, 'w', encoding='utf-8') as f:
        for s in samples:
            f.write(json.dumps(s) + '\n')
    print(f"Sample data written to {path}")


if __name__ == "__main__":
    # --- Step 1: create sample input ---
    create_sample_data("data.jsonl")

    # --- Step 2: run pipeline ---
    pipeline = MultiTurnPipeline(min_turns=3, max_turns=10)
    pipeline.run(input_path="data.jsonl", output_path="processed_dataset.jsonl")

    # --- Step 3: validate output ---
    print("\nValidating conversations...")
    with open("processed_dataset.jsonl", 'r') as f:
        for line in f:
            item = json.loads(line)
            ok, msg = validate_conversation(item['messages'])
            status = "✓" if ok else "✗"
            print(f"  [{status}] id={item['id']}  turns={len(item['messages'])}  validation={msg}")

    # --- Step 4: (Optional) tokenise with a real model ---
    # Uncomment and replace model name to actually tokenise:
    #
    # tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
    # dataset = MultiTurnSFTDataset("processed_dataset.jsonl", tokenizer, max_length=2048)
    # loader  = DataLoader(dataset, batch_size=2,
    #                      collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id))
    # batch   = next(iter(loader))
    # print("\nFirst batch shapes:", {k: v.shape for k, v in batch.items()})